# LangChain RAG Walkthrough with ChromaDB

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) implementation using:
- **LangChain** for orchestration
- **ChromaDB** as the vector store (ephemeral, no background processes)
- **Flexible LLM support** for OpenAI, Gemini, or Ollama
- **Document chunking** and **embedding** workflows
- **Similarity-based retrieval**

## Overview
1. Setup and Dependencies
2. Document Loading and Chunking
3. Embedding Generation
4. Vector Store Creation (ChromaDB)
5. Similarity Retrieval
6. LLM Integration (OpenAI/Gemini/Ollama)
7. Complete RAG Pipeline

In [2]:
# Install required packages
!pip install langchain langchain-community langchain-openai langchain-google-genai chromadb sentence-transformers pypdf2 python-dotenv bs4 langchain-ollama

In [3]:
# Import required libraries
import os
from typing import List, Optional
from dotenv import load_dotenv

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain_community.vectorstores import Chroma
from langchain.schema import BaseRetriever
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Embedding models
from langchain_openai import OpenAIEmbeddings

# LLM models
from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM

# Load environment variables
load_dotenv()

print("All libraries imported successfully!")

All libraries imported successfully!


## 1. Document Preparation and Chunking

We'll start by creating some sample documents and demonstrating how to chunk them effectively for RAG.

In [4]:
# Create sample documents for demonstration
sample_documents = [
    """
    Machine Learning is a subset of artificial intelligence that enables computers to learn and improve 
    from experience without being explicitly programmed. It focuses on the development of computer programs 
    that can access data and use it to learn for themselves. The process of learning begins with observations 
    or data, such as examples, direct experience, or instruction, in order to look for patterns in data and 
    make better decisions in the future based on the examples that we provide.
    """,
    """
    Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence "deep") 
    to model and understand complex patterns in data. These neural networks attempt to simulate the behavior 
    of the human brain, allowing it to "learn" from large amounts of data. Deep learning has been particularly 
    successful in areas such as image recognition, natural language processing, and speech recognition.
    """,
    """
    Natural Language Processing (NLP) is a branch of artificial intelligence that deals with the interaction 
    between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, 
    understand, and make sense of human languages in a manner that is valuable. NLP combines computational 
    linguistics with statistical, machine learning, and deep learning models to help computers process human language.
    """,
    """
    Retrieval-Augmented Generation (RAG) is an AI framework that combines the strengths of parametric and 
    non-parametric knowledge. It retrieves relevant information from a knowledge base and uses that information 
    to generate more accurate and contextually relevant responses. RAG models first retrieve relevant documents 
    from a corpus using a retriever, then use a generator to produce the final output conditioned on both the 
    query and the retrieved documents.
    """
]

# Convert to LangChain Document objects
documents = [Document(page_content=doc.strip(), metadata={"source": f"doc_{i}"}) 
             for i, doc in enumerate(sample_documents)]

print(f"Created {len(documents)} sample documents")
for i, doc in enumerate(documents):
    print(f"Document {i}: {len(doc.page_content)} characters")
    print(f"Preview: {doc.page_content[:100]}...")
    print("---")

Created 4 sample documents
Document 0: 508 characters
Preview: Machine Learning is a subset of artificial intelligence that enables computers to learn and improve ...
---
Document 1: 434 characters
Preview: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence ...
---
Document 2: 444 characters
Preview: Natural Language Processing (NLP) is a branch of artificial intelligence that deals with the interac...
---
Document 3: 478 characters
Preview: Retrieval-Augmented Generation (RAG) is an AI framework that combines the strengths of parametric an...
---


In [5]:
# Demonstrate different chunking strategies
def show_chunking_strategies():
    # Strategy 1: Small chunks
    small_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50,
        length_function=len,
    )
    
    # Strategy 2: Medium chunks
    medium_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=100,
        length_function=len,
    )
    
    # Strategy 3: Large chunks
    large_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        length_function=len,
    )
    
    # Test with the first document
    test_doc = documents[0]
    
    strategies = [
        ("Small (200 chars, 50 overlap)", small_splitter),
        ("Medium (400 chars, 100 overlap)", medium_splitter),
        ("Large (800 chars, 150 overlap)", large_splitter)
    ]
    
    for name, splitter in strategies:
        chunks = splitter.split_documents([test_doc])
        print(f"\n{name}:")
        print(f"Number of chunks: {len(chunks)}")
        for i, chunk in enumerate(chunks):
            print(f"  Chunk {i+1}: {len(chunk.page_content)} chars - '{chunk.page_content[:50]}...'")

show_chunking_strategies()


Small (200 chars, 50 overlap):
Number of chunks: 4
  Chunk 1: 99 chars - 'Machine Learning is a subset of artificial intelli...'
  Chunk 2: 103 chars - 'from experience without being explicitly programme...'
  Chunk 3: 105 chars - 'that can access data and use it to learn for thems...'
  Chunk 4: 183 chars - 'or data, such as examples, direct experience, or i...'

Medium (400 chars, 100 overlap):
Number of chunks: 2
  Chunk 1: 319 chars - 'Machine Learning is a subset of artificial intelli...'
  Chunk 2: 183 chars - 'or data, such as examples, direct experience, or i...'

Large (800 chars, 150 overlap):
Number of chunks: 1
  Chunk 1: 508 chars - 'Machine Learning is a subset of artificial intelli...'


## 2. Web Content Loading

LangChain provides powerful web loaders to extract content from websites. We'll demonstrate how to use the WebBaseLoader to load HTML content from websites and prepare it for RAG.

In [6]:
# Import web loading libraries
from typing import Optional
from langchain.docstore.document import Document
from langchain_community.document_loaders import WebBaseLoader

# Example of using WebBaseLoader to load a webpage
def load_webpage(url: str) -> Optional[Document]:
    try:
        loader = WebBaseLoader(url)
        documents = loader.load()
        if documents:
            return documents[0]  # Return the first document loaded
        else:
            print(f"No documents found for {url}")
            return None
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return None
# Example usage

web_documents = []

web_urls = ["https://en.wikipedia.org/wiki/Retrieval-augmented_generation", "https://rsecon25.society-rse.org"]

for url in web_urls:
    web_document = load_webpage(url)
    if web_document:
        print(f"Loaded webpage: {url}")
        web_documents.append(web_document)
    else:
        print(f"Failed to load webpage: {url}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded webpage: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Loaded webpage: https://rsecon25.society-rse.org


In [7]:


# Create final chunks for our RAG pipeline
# Using medium-sized chunks for optimal balance
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    length_function=len,
)

# Split all documents
all_chunks = text_splitter.split_documents(web_documents)

print(f"✅ Created {len(all_chunks)} chunks from {len(documents)} documents")
print("\nChunk details:")
for i, chunk in enumerate(all_chunks):
    source = chunk.metadata.get('source', 'unknown')
    print(f"Chunk {i+1}: {len(chunk.page_content)} chars from {source}")

✅ Created 125 chunks from 4 documents

Chunk details:
Chunk 1: 367 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 2: 376 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 3: 370 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 4: 351 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 5: 365 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 6: 383 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 7: 131 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 8: 40 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 9: 397 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 10: 260 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 11: 396 chars from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Chunk 12: 

## 2. Embedding Strategies

We'll demonstrate different embedding approaches that work with various providers.

In [8]:
# Configure different embedding models
def get_embedding_model(provider: str = "ollama"):
    """
    Get embedding model based on provider choice.
    Options: "openai", "ollama"
    """
    if provider == "openai":
        # Requires OPENAI_API_KEY environment variable
        return OpenAIEmbeddings(model="text-embedding-3-small")
    
    elif provider == "ollama":
        # Free, local embedding model using Ollama (no API key required)
        # Requires: ollama pull nomic-embed-text
        from langchain_community.embeddings import OllamaEmbeddings
        return OllamaEmbeddings(
            model="nomic-embed-text",
            base_url="http://localhost:11434"
        )
    
    else:
        raise ValueError(f"Unsupported provider: {provider}. Use 'openai' or 'ollama'")

# Demonstrate embedding creation
print("Available embedding providers:")
print("1. ollama (free, local - requires: ollama pull nomic-embed-text)")
print("2. openai (requires OPENAI_API_KEY)")

# Use Ollama by default (no API key required, but requires ollama setup)
embeddings = get_embedding_model("ollama")
print(f"\n Using Ollama embeddings: {embeddings}")

Available embedding providers:
1. ollama (free, local - requires: ollama pull nomic-embed-text)
2. openai (requires OPENAI_API_KEY)

 Using Ollama embeddings: base_url='http://localhost:11434' model='nomic-embed-text' embed_instruction='passage: ' query_instruction='query: ' mirostat=None mirostat_eta=None mirostat_tau=None num_ctx=None num_gpu=None num_thread=None repeat_last_n=None repeat_penalty=None temperature=None stop=None tfs_z=None top_k=None top_p=None show_progress=False headers=None model_kwargs=None


/tmp/ipykernel_2937987/967169450.py:15: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  return OllamaEmbeddings(


## 3. ChromaDB Vector Store

Now we'll create an ephemeral ChromaDB vector store and embed our documents.

In [9]:
# Create ChromaDB vector store with persistence
import os

# Define the directory where ChromaDB will save the vector store
persist_directory = "./chromadb_storage"

print(f"Creating ChromaDB vector store with persistence at: {persist_directory}")

# Create the vector store with our chunks and embeddings
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="rag_demo",
    persist_directory=persist_directory  # This enables persistence to disk
)

# The vector store is automatically saved to disk when created
print(f"✅ Vector store created and saved to {persist_directory}")
print(f"Vector store contains {vectorstore._collection.count()} documents")

# Verify the embedding process worked
collection_info = vectorstore._collection.count()
print(f"Collection contains {collection_info} embedded documents")

# Show some collection metadata
try:
    # Peek at the collection to verify it's working
    peek_result = vectorstore._collection.peek(limit=2)
    print(f"Sample IDs: {peek_result['ids'][:2] if peek_result['ids'] else 'None'}")
    print(f"Sample metadata keys: {list(peek_result['metadatas'][0].keys()) if peek_result['metadatas'] else 'None'}")
except Exception as e:
    print(f"Note: {e}")

# Show what was saved to disk
if os.path.exists(persist_directory):
    files = os.listdir(persist_directory)
    print(f"Files saved to disk: {files}")
else:
    print("Warning: Persist directory not found")

Creating ChromaDB vector store with persistence at: ./chromadb_storage
✅ Vector store created and saved to ./chromadb_storage
Vector store contains 352 documents
Collection contains 352 embedded documents
Sample IDs: ['efe05a57-6c48-445b-8ce2-233309229787', '1d8acf32-3f56-4b56-9b33-c87ed2da2c5b']
Sample metadata keys: ['source', 'language', 'title']
Files saved to disk: ['35804d8d-2f19-4d4a-b809-6d5df344f81d', 'chroma.sqlite3']


In [10]:
# Loading an existing ChromaDB vector store from disk
def load_existing_vectorstore(persist_directory: str = "./chromadb_storage", collection_name: str = "rag_demo"):
    """
    Load an existing ChromaDB vector store from disk.
    
    Args:
        persist_directory: Directory where the vector store is saved
        collection_name: Name of the collection to load
    
    Returns:
        Loaded Chroma vector store
    """
    if not os.path.exists(persist_directory):
        raise FileNotFoundError(f"Vector store directory not found: {persist_directory}")
    
    print(f"Loading existing vector store from: {persist_directory}")
    
    # Load the existing vector store
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,  # Must use the same embedding function
        collection_name=collection_name
    )
    
    print(f"✅ Loaded vector store with {vectorstore._collection.count()} documents")
    return vectorstore

# Example usage (commented out to avoid overwriting the current vectorstore):
# loaded_vectorstore = load_existing_vectorstore()
# print(f"Loaded vector store contains {loaded_vectorstore._collection.count()} documents")

print("💾 Vector store persistence is now enabled!")
print("You can now:")
print("1. Close this notebook and the vector store will remain saved")
print("2. Use load_existing_vectorstore() to reload it in future sessions")
print("3. Share the chromadb_storage folder to share your vector store")

💾 Vector store persistence is now enabled!
You can now:
1. Close this notebook and the vector store will remain saved
2. Use load_existing_vectorstore() to reload it in future sessions
3. Share the chromadb_storage folder to share your vector store


## 4. Similarity Retrieval

Let's test the retrieval functionality with different queries and similarity thresholds.

In [11]:
# Demonstrate similarity search
def test_similarity_search(query: str, k: int = 3):
    """Test similarity search with a given query"""
    print(f"\n🔍 Query: '{query}'")
    print(f"Retrieving top {k} similar documents:")
    print("-" * 50)
    
    # Similarity search
    results = vectorstore.similarity_search(query, k=k)
    
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get('source', 'unknown')
        print(f"{i}. Source: {source}")
        print(f"   Content: {doc.page_content[:150]}...")
        print()
    
    return results

# Test with different queries
test_queries = [
    "What is machine learning?",
    "neural networks and deep learning",
    "How does RAG work?",
    "natural language processing applications"
]

for query in test_queries:
    test_similarity_search(query, k=2)


🔍 Query: 'What is machine learning?'
Retrieving top 2 similar documents:
--------------------------------------------------
1. Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: Reflection
Recursive self-improvement
Hallucination
Word embedding
Vibe coding
Applications
Machine learning
In-context learning
Artificial neural net...

2. Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: Reflection
Recursive self-improvement
Hallucination
Word embedding
Vibe coding
Applications
Machine learning
In-context learning
Artificial neural net...


🔍 Query: 'neural networks and deep learning'
Retrieving top 2 similar documents:
--------------------------------------------------
1. Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: Transformer
Vision transformer (ViT)
Recurrent neural network (RNN)
Long short-term memory (LSTM)
Gated recurrent unit (GRU)
Echo state network
Multil...

2. Source: https://en.wik

In [12]:
# Demonstrate similarity search with scores
def test_similarity_search_with_scores(query: str, k: int = 3):
    """Test similarity search with similarity scores"""
    print(f"\n📊 Query with scores: '{query}'")
    print("-" * 50)
    
    # Similarity search with scores
    results = vectorstore.similarity_search_with_score(query, k=k)
    
    for i, (doc, score) in enumerate(results, 1):
        source = doc.metadata.get('source', 'unknown')
        print(f"{i}. Similarity Score: {score:.4f}")
        print(f"   Source: {source}")
        print(f"   Content: {doc.page_content[:100]}...")
        print()
    
    return results

# Test with scores
test_similarity_search_with_scores("What is deep learning?", k=15)

# Test retriever functionality
print("\n🔄 Testing retriever interface:")
retriever = vectorstore.as_retriever(search_kwargs={"k": 15})
retrieved_docs = retriever.get_relevant_documents("machine learning algorithms")

print(f"Retrieved {len(retrieved_docs)} documents using retriever interface")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content[:80]}...")


📊 Query with scores: 'What is deep learning?'
--------------------------------------------------
1. Similarity Score: 362.1430
   Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: vteGenerative AIConcepts
Autoencoder
Deep learning
Fine-tuning
Foundation model
Generative adversari...

2. Similarity Score: 362.1430
   Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: vteGenerative AIConcepts
Autoencoder
Deep learning
Fine-tuning
Foundation model
Generative adversari...

3. Similarity Score: 362.1430
   Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: vteGenerative AIConcepts
Autoencoder
Deep learning
Fine-tuning
Foundation model
Generative adversari...

4. Similarity Score: 391.9182
   Source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
   Content: Reflection
Recursive self-improvement
Hallucination
Word embedding
Vibe coding
Applications
Machine ...

5. Similarity Score: 391.9

/tmp/ipykernel_2937987/3305061829.py:25: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents("machine learning algorithms")


## 5. LLM Integration (OpenAI/Gemini/Ollama)

Now we'll set up flexible LLM integration that can work with different providers.

In [13]:
# Configure different LLM providers
def get_llm(provider: str = "ollama", model: str = None):
    """
    Get LLM based on provider choice.
    Options: "openai", "ollama"
    """
    if provider == "openai":
        # Requires OPENAI_API_KEY environment variable
        model = model or "gpt-3.5-turbo"
        return ChatOpenAI(
            model=model,
            temperature=0.1,
            max_tokens=1000
        )
    
    elif provider == "ollama":
        # Requires Ollama to be running locally
        # Install: https://ollama.ai/
        # Run: ollama pull llama3.2:1b (or your preferred model)
        model = model or "llama3.2:1b"  # Using the lightweight 1B parameter model
        return OllamaLLM(
            model=model,
            temperature=0.1
        )
    
    else:
        raise ValueError(f"Unsupported provider: {provider}. Use 'openai' or 'ollama'")

## 6. Complete RAG Pipeline

Now let's put it all together into a complete RAG system!

In [17]:
# Complete RAG Pipeline Class
class RAGPipeline:
    def __init__(self, embedding_provider="ollama", llm_provider="ollama", llm_model=None):
        """
        Initialize RAG pipeline with flexible provider options.
        
        Args:
            embedding_provider: "ollama" or "openai"
            llm_provider: "ollama" or "openai"  
            llm_model: Specific model name (optional)
        """
        self.embedding_provider = embedding_provider
        self.llm_provider = llm_provider
        
        # Initialize embeddings
        self.embeddings = get_embedding_model(embedding_provider)
        
        # Initialize LLM
        try:
            self.llm = get_llm(llm_provider, llm_model)
        except Exception as e:
            print(f"Warning: Could not initialize LLM ({llm_provider}): {e}")
            self.llm = None
        
        self.vectorstore = None
        self.retriever = None
        self.qa_chain = None
        
    def create_vectorstore(self, documents, collection_name="rag_collection", persist_directory=None):
        """
        Create ChromaDB vector store from documents.
        
        Args:
            documents: List of documents to embed
            collection_name: Name for the ChromaDB collection
            persist_directory: Directory to save the vector store (None for ephemeral)
        """
        print(f"Creating vector store with {len(documents)} documents...")
        
        if persist_directory:
            print(f"Vector store will be saved to: {persist_directory}")
            self.vectorstore = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                collection_name=collection_name,
                persist_directory=persist_directory
            )
        else:
            print("Creating ephemeral (in-memory) vector store")
            self.vectorstore = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                collection_name=collection_name
            )
        
        # Create retriever
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}
        )
        
        print(f"✅ Vector store created with {self.vectorstore._collection.count()} documents")
        return self.vectorstore
    
    def load_vectorstore(self, persist_directory, collection_name="rag_collection"):
        """
        Load an existing ChromaDB vector store from disk.
        
        Args:
            persist_directory: Directory where the vector store is saved
            collection_name: Name of the collection to load
        """
        if not os.path.exists(persist_directory):
            raise FileNotFoundError(f"Vector store directory not found: {persist_directory}")
        
        print(f"Loading existing vector store from: {persist_directory}")
        
        self.vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name=collection_name
        )
        
        # Create retriever
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}
        )
        
        print(f"✅ Loaded vector store with {self.vectorstore._collection.count()} documents")
        return self.vectorstore
    
    def setup_qa_chain(self):
        """Set up the question-answering chain."""
        if not self.llm:
            raise ValueError("LLM not available. Please configure an LLM provider.")
        
        if not self.retriever:
            raise ValueError("Retriever not available. Please create vector store first.")
        
        # Custom prompt template
        prompt_template = """Use the following pieces of context to answer the question at the end. 
        If you don't know the answer, just say that you don't know, don't try to make up an answer.

        Context:
        {context}

        Question: {question}
        
        Answer:"""
        
        PROMPT = PromptTemplate(
            template=prompt_template,
            input_variables=["context", "question"]
        )
        
        # Create QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=self.retriever,
            chain_type_kwargs={"prompt": PROMPT},
            return_source_documents=True
        )
        
        print("✅ QA chain configured")
        return self.qa_chain
    
    def query(self, question: str, return_sources: bool = True):
        """Query the RAG system."""
        if not self.qa_chain:
            raise ValueError("QA chain not configured. Please run setup_qa_chain() first.")
        
        print(f"\n🤖 Question: {question}")
        print("-" * 50)
        
        result = self.qa_chain({"query": question})
        
        answer = result["result"]
        sources = result.get("source_documents", [])
        
        print(f"Answer: {answer}")
        
        if return_sources and sources:
            print(f"\n📚 Sources ({len(sources)} documents):")
            for i, doc in enumerate(sources, 1):
                source = doc.metadata.get('source', 'unknown')
                print(f"{i}. {source}: {doc.page_content[:100]}...")
        
        return {
            "question": question,
            "answer": answer,
            "sources": sources
        }

# Initialize the RAG pipeline
rag = RAGPipeline(
    embedding_provider="ollama",  # Local Ollama embeddings
    llm_provider="ollama",  # Change to "openai" if you have API key
    llm_model="llama3.2:1b"  # Using the lightweight 1B parameter model
)


In [21]:
# Set up the complete RAG system with persistence
print("Setting up complete RAG system with persistent storage...")

# Create vector store with our chunks and save to disk
rag.create_vectorstore(
    documents=all_chunks, 
    collection_name="ai_knowledge_base",
    persist_directory="./rag_vectorstore"  # Save to disk
)

rag.setup_qa_chain()

print("💾 RAG system is now set up with persistent storage!")
print("The vector store is saved to './rag_vectorstore' directory")

# Alternative: Load existing vector store instead of creating new one
# Uncomment the following lines if you want to load an existing vector store:
# 
# rag.load_vectorstore(
#     persist_directory="./rag_vectorstore",
#     collection_name="ai_knowledge_base"
# )
# rag.setup_qa_chain()

Setting up complete RAG system with persistent storage...
Creating vector store with 125 documents...
Vector store will be saved to: ./rag_vectorstore
✅ Vector store created with 250 documents
✅ QA chain configured
💾 RAG system is now set up with persistent storage!
The vector store is saved to './rag_vectorstore' directory


In [22]:
# Test the complete RAG system
if rag.llm and rag.qa_chain:
    print("🧪 Testing complete RAG pipeline...")
    
    # Test queries
    test_questions = [
        "What is machine learning?",
        "How does deep learning differ from traditional machine learning?", 
        "What are the main applications of NLP?",
        "Explain how RAG works"
        "What is rsecon25?"
    ]
    
    for question in test_questions:
        try:
            result = rag.query(question)
            print("\n" + "="*60 + "\n")
        except Exception as e:
            print(f"Error processing question '{question}': {e}")
            break
            
else:
    print("🔍 Testing retrieval functionality only...")
    
    # Test retrieval without LLM
    if rag.retriever:
        test_questions = [
            "What is machine learning?",
            "deep learning neural networks",
            "natural language processing"
        ]
        
        for question in test_questions:
            print(f"\n🔍 Retrieving for: '{question}'")
            docs = rag.retriever.get_relevant_documents(question)
            for i, doc in enumerate(docs, 1):
                source = doc.metadata.get('source', 'unknown')
                print(f"{i}. {source}: {doc.page_content[:100]}...")
            print("-" * 40)

🧪 Testing complete RAG pipeline...

🤖 Question: What is machine learning?
--------------------------------------------------
Answer: I don't know.

📚 Sources (3 documents):
1. https://en.wikipedia.org/wiki/Retrieval-augmented_generation: Reflection
Recursive self-improvement
Hallucination
Word embedding
Vibe coding
Applications
Machine ...
2. https://en.wikipedia.org/wiki/Retrieval-augmented_generation: Reflection
Recursive self-improvement
Hallucination
Word embedding
Vibe coding
Applications
Machine ...
3. https://en.wikipedia.org/wiki/Retrieval-augmented_generation: vteGenerative AIConcepts
Autoencoder
Deep learning
Fine-tuning
Foundation model
Generative adversari...



🤖 Question: How does deep learning differ from traditional machine learning?
--------------------------------------------------
Answer: I don't know, as I'm not familiar with the specific differences between deep learning and traditional machine learning. Can you provide more context or information about what you me